# SPM 3405: Machine Learning — Lecture 1 notebook
## Introduction to machine learning

**Dr Charles Ndung'u** · Department of Physics, University of Nairobi

In this notebook you will:
1. Store data as a matrix $\mathbf{X}$ and make predictions with $\hat{\mathbf{y}} = \mathbf{X}\boldsymbol\beta$.
2. Fit polynomials of increasing flexibility to cooling-curve data and compare **training MSE** with **test MSE**.
3. Build a **$K$-nearest neighbours** classifier from scratch to tell healthy machines from faulty ones, and see how $K$ controls the bias–variance trade-off.

Run each cell with **Shift + Enter**, from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(405)   # fixed seed: everyone gets the same numbers

## Part 1: Data as a matrix

Solar panel measurements: irradiance $x_{i1}$ (W/m²), cell temperature $x_{i2}$ (°C) and output power $y_i$ (W).
Each **row** is one observation; each **column** is one variable.

In [ ]:
X = np.array([[260, 31.1],
              [270, 31.2],
              [270, 21.9],
              [950, 60.4]])
y = np.array([70.3, 86.4, 78.5, 231.4])

n, p = X.shape
print("n =", n, " p =", p)
print("X transpose has shape", X.T.shape)

# A linear model without intercept: y_hat = X @ beta
beta = np.array([0.24, -0.2])      # a guess for the coefficients
y_hat = X @ beta
print("predictions:", y_hat.round(1))
print("true values:", y)

**Try it:** change `beta` to get predictions closer to `y`. How would you choose `beta` *systematically*? (That is next lecture.)

## Part 2: Training MSE versus test MSE

A temperature sensor sits in a cup of tea that cools according to Newton's law, $T(t) = 25 + 60\,e^{-t/8}$ (°C, with $t$ in minutes).
The sensor has random noise with $\sigma = 2$ °C, so we observe $Y = f(t) + \varepsilon$.

We take **15 training readings** and keep **1000 test readings** that the model never sees during fitting.

In [ ]:
def f(t):
    return 25 + 60 * np.exp(-t / 8)

sigma = 2.0      # noise level (degrees C)
n_train = 15

t_train = np.sort(rng.uniform(0.5, 29.5, n_train))
T_train = f(t_train) + rng.normal(0, sigma, n_train)
t_test = rng.uniform(t_train.min(), t_train.max(), 1000)
T_test = f(t_test) + rng.normal(0, sigma, t_test.size)

grid = np.linspace(t_train.min(), t_train.max(), 300)
plt.plot(grid, f(grid), "k--", label="true f(t)")
plt.plot(t_train, T_train, "o", label="training readings")
plt.xlabel("t (min)"); plt.ylabel("T (°C)"); plt.legend(); plt.show()

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

degrees = range(1, 13)
train_mse, test_mse = [], []
for d in degrees:
    model = np.polynomial.Polynomial.fit(t_train, T_train, d)   # least-squares polynomial fit
    train_mse.append(mse(T_train, model(t_train)))
    test_mse.append(mse(T_test, model(t_test)))

plt.semilogy(degrees, train_mse, "o-", label="training MSE")
plt.semilogy(degrees, test_mse, "s-", label="test MSE")
plt.axhline(sigma**2, ls="--", color="green", label="Var(ε) (irreducible)")
plt.xlabel("flexibility (polynomial degree)"); plt.ylabel("MSE (°C²)"); plt.legend(); plt.show()

best = list(degrees)[int(np.argmin(test_mse))]
print("Degree with the lowest test MSE:", best)

In [ ]:
plt.figure(figsize=(12, 3.5))
for k, d in enumerate([1, best, 12]):
    model = np.polynomial.Polynomial.fit(t_train, T_train, d)
    plt.subplot(1, 3, k + 1)
    plt.plot(t_train, T_train, "o")
    plt.plot(grid, model(grid), "r-")
    plt.plot(grid, f(grid), "k--")
    plt.ylim(10, 100); plt.title(f"degree {d}"); plt.xlabel("t (min)")
plt.tight_layout(); plt.show()

**Questions**
1. Which degree **underfits** (high bias) and which **overfits** (high variance)? Explain using the plots.
2. Why can the test MSE never go below the green line?
3. Change `n_train` to `100` and run Part 2 again. What happens to the best degree and to the gap between training and test MSE?
4. Change `sigma` to `5`. What changes?

## Part 3: Classification with $K$-nearest neighbours

A monitoring system records two standardised sensor features for each machine: vibration level $x_1$ and temperature $x_2$.
**Healthy** machines cluster around $(0, 0)$; **faulty** machines cluster around $(1.5, 1.5)$.
Both classes are equally likely, so here we *know* the Bayes classifier: faulty if $x_1 + x_2 > 1.5$.

In [ ]:
def make_machines(n):
    labels = rng.integers(0, 2, n)                       # 0 = healthy, 1 = faulty
    centres = np.where(labels[:, None] == 1, 1.5, 0.0)
    return centres + rng.normal(0, 1, (n, 2)), labels

X_train, y_train = make_machines(200)
X_test, y_test = make_machines(2000)

bayes_pred = (X_test.sum(axis=1) > 1.5).astype(int)
bayes_error = np.mean(bayes_pred != y_test)
print(f"Bayes error rate (the best possible): {bayes_error:.3f}")

plt.scatter(*X_train[y_train == 0].T, s=15, label="healthy")
plt.scatter(*X_train[y_train == 1].T, s=15, label="faulty")
plt.plot([-3, 4.5], [4.5, -3], "k--", label="Bayes boundary")
plt.xlabel("vibration $x_1$"); plt.ylabel("temperature $x_2$"); plt.legend(); plt.axis("equal"); plt.show()

KNN estimates $\Pr(Y = j \mid X = x_0) \approx \frac{1}{K}\sum_{i \in \mathcal{N}_0} I(y_i = j)$ and picks the most common class among the $K$ nearest training points.

In [ ]:
def knn_predict(X_train, y_train, X_new, K):
    # distance from every new point to every training point
    dist = np.linalg.norm(X_new[:, None, :] - X_train[None, :, :], axis=2)
    nearest = np.argsort(dist, axis=1)[:, :K]               # indices of the K nearest
    prob_faulty = y_train[nearest].mean(axis=1)             # fraction of neighbours that are faulty
    return (prob_faulty > 0.5).astype(int)

Ks = [1, 3, 5, 11, 21, 51, 101]
train_err = [np.mean(knn_predict(X_train, y_train, X_train, K) != y_train) for K in Ks]
test_err = [np.mean(knn_predict(X_train, y_train, X_test, K) != y_test) for K in Ks]

plt.semilogx(Ks, train_err, "o-", label="training error")
plt.semilogx(Ks, test_err, "s-", label="test error")
plt.axhline(bayes_error, ls="--", color="green", label="Bayes error")
plt.xlabel("K (large K = less flexible)"); plt.ylabel("error rate"); plt.legend(); plt.show()

for K, e in zip(Ks, test_err):
    print(f"K = {K:3d}: test error = {e:.3f}")

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-3, 4.5, 150), np.linspace(-3, 4.5, 150))
grid_points = np.c_[g1.ravel(), g2.ravel()]

plt.figure(figsize=(10, 4))
for k, K in enumerate([1, 101]):
    region = knn_predict(X_train, y_train, grid_points, K).reshape(g1.shape)
    plt.subplot(1, 2, k + 1)
    plt.contourf(g1, g2, region, alpha=0.25, levels=[-0.5, 0.5, 1.5])
    plt.scatter(*X_train[y_train == 0].T, s=10)
    plt.scatter(*X_train[y_train == 1].T, s=10)
    plt.plot([-3, 4.5], [4.5, -3], "k--")
    plt.title(f"KNN, K = {K}"); plt.xlabel("vibration $x_1$"); plt.ylabel("temperature $x_2$")
plt.tight_layout(); plt.show()

**Questions**
1. Why is the training error for $K = 1$ exactly zero? Is that a good model?
2. Which $K$ gives the lowest test error? How close does it get to the Bayes error?
3. Describe the $K = 1$ and $K = 101$ boundaries in terms of bias and variance.
4. **Your group project:** name one instrumentation problem where you would use classification. Which sensor readings would $x_1, x_2, \dots$ be?